# **Minimax-H3 for AI Video Generation (ComfyUI)**
- Run the cell below to get a link (e.g. https://literature-consortium-align.trycloudflare.com ) which you can use to launch the comfyUI interface.
- If you get an error on the resize image node, change the upscale_method.
- Notebook source: https://github.com/Isi-dev/Google-Colab_Notebooks
- No Civitai support, only Huggingface links.

In [8]:
# @title # 1. 💥 Prepare Environment & Install Dependencies {"single-column":true}
import os

# 1. ALWAYS force the directory back to the Colab root before cloning
%cd /content

# 2. Clone ComfyUI only if it doesn't already exist in the root
if not os.path.exists("ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI.git

%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/larryvrh/ComfyUI-MiniMax-H3-Turbo

# 3. Move into the correct, top-level ComfyUI folder
%cd /content/ComfyUI

# Install core dependencies
!pip install -r requirements.txt
from IPython.display import clear_output

# 4. Install custom nodes only if they don't already exist
if not os.path.exists("custom_nodes/ComfyUI-GGUF"):
    !cd custom_nodes && git clone https://github.com/city96/ComfyUI-GGUF.git

# 5. Install Video Helper Suite
if not os.path.exists("custom_nodes/ComfyUI-VideoHelperSuite"):
    !cd custom_nodes && git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git
    !cd custom_nodes/ComfyUI-VideoHelperSuite && pip install -r requirements.txt

# 6. Install KJNodes
if not os.path.exists("custom_nodes/ComfyUI-KJNodes"):
    !cd custom_nodes && git clone https://github.com/kijai/ComfyUI-KJNodes.git
    !cd custom_nodes/ComfyUI-KJNodes && pip install -r requirements.txt

# Install GGUF and high-speed Hugging Face download engine
!pip install gguf huggingface_hub hf_transfer sageattention triton

clear_output()
# Upgrade to prevent limit for larger files
!pip install --upgrade huggingface_hub

# @markdown Select your desired models below.

# @markdown ---
# @markdown ### **HuggingFace Token (Optional but Recommended)**
# @markdown Adding a free token prevents rate-limiting and stalling on massive downloads. Get yours at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens). Files larger than 14GB may still take a while to download.
hf_token = "" # @param {type:"string"}

# @markdown ---
# @markdown ### **VAEs**
vaes_repo = "Comfy-Org/MiniMax-H3"  # @param {type:"string"}
audio_vae = "vae/minimax_h3_audio_vae_fp32.safetensors"  # @param {type:"string"}
video_vae = "vae/minimax_h3_video_vae_fp16.safetensors" # @param {type:"string"}

# @markdown ---
# @markdown ### **Text Encoders**
text_encoder_repo = "Comfy-Org/MiniMax-H3"  # @param {type:"string"}
text_encoder_model = "text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors" # @param {type:"string"}

download_audio_vae = True
download_video_vae = True

# @markdown ---
# @markdown ### **UNet Models**
unet_repo = "Comfy-Org/MiniMax-H3"  # @param {type:"string"}
unet_model = "diffusion_models/minimax_h3_ref2va_pruned_fp8_scaled.safetensors" # @param {type:"string"}
# @markdown ---
# @markdown ### **Turbo LoRa**
turbo_repo = "larryvrh/MiniMax-H3-Turbo-Lora"  # @param {type:"string"}
turbo_lora = "minimax_h3_turbo_v4_step600_ema.safetensors"  # @param {type:"string"}
# @markdown ---
# @markdown ### **LoRa 1**
lora1_repo = ""  # @param {type:"string"}
lora1 = ""  # @param {type:"string"}
# @markdown ---
# @markdown ### **LoRa 2**
lora2_repo = ""  # @param {type:"string"}
lora2 = ""  # @param {type:"string"}
# @markdown ---
# @markdown ### **LoRa 3**
lora3_repo = ""  # @param {type:"string"}
lora3 = ""  # @param {type:"string"}


import os
from pathlib import Path
from huggingface_hub import hf_hub_download

# Ensure environment is set to ComfyUI root directory
%cd /content/ComfyUI

# Enable Rust-accelerated fast transfers
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

import shutil

def download_filec(repo_id, repo_relative_path, local_destination, token=None):
    if not repo_relative_path or repo_relative_path == "None":
        return False

    os.makedirs(local_destination, exist_ok=True)

    filename = os.path.basename(repo_relative_path)
    destination = os.path.join(local_destination, filename)

    if os.path.isfile(destination):
        print(f"⏭️ Already exists: {destination}")
        return True

    print(f"⬇️ Downloading:")
    print(f"   Repo: {repo_id}")
    print(f"   File: {repo_relative_path}")
    print(f"   To:   {destination}")

    try:
        downloaded_path = hf_hub_download(
            repo_id=repo_id,
            filename=repo_relative_path,
            token=token,
        )

        shutil.copy2(downloaded_path, destination)
        print(f"downloaded path --- {downloaded_path}")
        os.remove(downloaded_path)

        if os.path.isfile(downloaded_path):
            print(f"⏭️ Temporary  file couldn't be deleted: {downloaded_path}")
        else:
            print(f"⏭️ Temporary  file deleted: {downloaded_path}")

        print(f"✅ Download complete:")
        print(f"   {destination}")
        print(f"   Size: {os.path.getsize(destination):,} bytes\n")

        return True

    except Exception as e:
        print(f"❌ Download failed:")
        print(f"   {repo_id}/{repo_relative_path}")
        print(f"   {type(e).__name__}: {e}\n")

        return False

# Organized by file size for huggingface
token_str = hf_token.strip()
if turbo_lora != "":
    download_filec(turbo_repo, f"{turbo_lora}", f"models/loras/", token_str)
!hf cache rm --yes "model/{turbo_repo}"
if lora1 != "":
    download_filec(lora1_repo, f"{lora1}", f"models/loras/", token_str)
!hf cache rm --yes "model/{lora1_repo}"
if lora2 != "":
    download_filec(lora2_repo, f"{lora2}", f"models/loras/", token_str)
!hf cache rm --yes "model/{lora2_repo}"
if lora3 != "":
    download_filec(lora3_repo, f"{lora3}", f"models/loras/", token_str)
!hf cache rm --yes "model/{lora3_repo}"
download_filec(vaes_repo, f"{audio_vae}", f"models/vae/", token_str)
download_filec(vaes_repo, f"{video_vae}", f"models/vae/", token_str)
!hf cache rm --yes "model/{vaes_repo}"
download_filec(unet_repo, f"{unet_model}", f"models/diffusion_models/", token_str)
!hf cache rm --yes "model/{unet_repo}"
download_filec(text_encoder_repo, f"{text_encoder_model}", f"models/text_encoders/", token_str)
!hf cache rm --yes "model/{text_encoder_repo}"

#Making sure that the cache is clear
print(f"Listing HF cache")
!hf cache ls
print(f"Deleting incomplete the HF downloads in the cache")
!hf cache prune
print(f"The HF cache is clear.")

clear_output()

# @title 3. 🚀 Run ComfyUI
use_cloudflare = False # @param {type:"boolean"}
use_interface_in_cell = False # @param {type:"boolean"}

# @markdown ### **VRAM Management**
# @markdown Select how you want ComfyUI to handle model loading:
# @markdown - **High VRAM (Keep loaded):** Keeps models in VRAM for faster subsequent generation.
# @markdown - **Normal VRAM (Load/Unload):** Offloads models from VRAM after use to save memory. (Default)
vram_management = "Normal VRAM (Load/Unload)" # @param ["Normal VRAM (Load/Unload)", "High VRAM (Keep loaded)"]

import torch
import os
from IPython.display import clear_output

clear_output()

%cd /content/ComfyUI

# Configure Launch Args based on selection
launch_args = "--enable-cors-header"

if vram_management == "High VRAM (Keep loaded)":
    launch_args += " --highvram"
# Note: ComfyUI uses "Normal VRAM" behavior by default, so we don't need to add any flags for it.

if use_cloudflare:
    if not os.path.exists("cloudflared-linux-amd64.deb"):
        !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
        !dpkg -i cloudflared-linux-amd64.deb

    import subprocess
    import threading
    import time
    import socket

    def iframe_thread(port):
        while True:
            time.sleep(0.5)
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            result = sock.connect_ex(('127.0.0.1', port))
            if result == 0:
                break
            sock.close()
        print("\nComfyUI finished loading, launching Cloudflare tunnel...\n")

        p = subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        for line in p.stderr:
            l = line.decode()
            if "trycloudflare.com " in l:
                print("This is your ComfyUI URL:", l[l.find("http"):], end='')
        clear_output()

    threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

    !python main.py $launch_args

elif use_interface_in_cell:
    import threading
    import time
    import socket

    def iframe_thread(port):
        while True:
            time.sleep(0.5)
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            result = sock.connect_ex(('127.0.0.1', port))
            if result == 0:
                break
            sock.close()
        from google.colab import output
        output.serve_kernel_port_as_iframe(port, height=1024)
        clear_output()
        print("To open in a standalone window click here:")
        output.serve_kernel_port_as_window(port)

    threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

    !python main.py $launch_args

else:
    import socket, time, threading
    from google.colab import output

    def link_thread(port):
        while True:
            time.sleep(0.5)
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            result = sock.connect_ex(('127.0.0.1', port))
            if result == 0:
                break
            sock.close()
        #clear_output()
        print("Click the link below to launch the ComfyUI interface:")
        output.serve_kernel_port_as_window(port)

    threading.Thread(target=link_thread, daemon=True, args=(8188,)).start()

    !python main.py $launch_args

/content/ComfyUI
⏭️ Already exists: models/loras/minimax_h3_turbo_v4_step600_ema.safetensors
/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
  - model/larryvrh/MiniMax-H3-Turbo-Lora
Nothing to delete.
/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  

text_encoders/qwen3vl_32b_minimax_h3_nvf(…): reconstructing file:   0%|          |  0.00B / 15.7GB            

text_encoders/qwen3vl_32b_minimax_h3_nvf(…): downloading bytes:           |  0.00B            

downloaded path --- /root/.cache/huggingface/hub/models--Comfy-Org--MiniMax-H3/snapshots/6701b0a14feefd7141bd9cfe8386961c27007622/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors
⏭️ Temporary  file deleted: /root/.cache/huggingface/hub/models--Comfy-Org--MiniMax-H3/snapshots/6701b0a14feefd7141bd9cfe8386961c27007622/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors
✅ Download complete:
   models/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors
   Size: 15,687,142,551 bytes

/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
About to delete 1 repo(s) totalling 0.0.
  - model/Comfy-Org/M

<IPython.core.display.Javascript object>

[INFO] Using RAM pressure cache.
[INFO] Starting server

[INFO] To see the GUI go to: http://127.0.0.1:8188
[INFO] 
Stopped server
